# 04 — External Demand Drivers & Leading Indicators

**Reference:** Vandeput, N. (2021). *Data Science for Supply Chain Forecasting* (2nd ed.), Chapter 20.

This is the notebook where supply chain DS becomes genuinely powerful.
External demand drivers are signals from outside the demand time series itself
that help predict future demand.

**For an energy drink company like Red Bull, relevant drivers include:**

| Driver | Signal | Source |
|--------|--------|--------|
| Temperature | Hot weather → higher energy drink demand | Open-Meteo API |
| Sporting events | Major events → demand spike in host region | Public events calendars |
| Google Trends | Search interest in 'energy drink' | pytrends |
| Public holidays | Weekend/holiday demand patterns | pandas holiday calendars |
| Music festivals | Festival season → on-premise demand surge | Scraped events data |

**This notebook demonstrates the methodology using simulated + real open data.**


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

## 1. Fetch Real Weather Data — Open-Meteo API

Free, no API key required. We fetch temperature data for Salzburg, Austria.

In [ ]:
def fetch_weather_salzburg(start_date='2019-01-01', end_date='2024-01-01'):
    """
    Fetch weekly average temperature for Salzburg from Open-Meteo.
    Free API — no key required.
    """
    url = 'https://archive-api.open-meteo.com/v1/archive'
    params = {
        'latitude': 47.80,
        'longitude': 13.04,
        'start_date': start_date,
        'end_date': end_date,
        'daily': 'temperature_2m_mean',
        'timezone': 'Europe/Vienna'
    }
    response = requests.get(url, params=params, timeout=30)
    data = response.json()

    df = pd.DataFrame({
        'date': pd.to_datetime(data['daily']['time']),
        'temp_c': data['daily']['temperature_2m_mean']
    }).set_index('date')

    # Resample to weekly
    return df.resample('W').mean()


print('Fetching weather data for Salzburg...')
try:
    weather = fetch_weather_salzburg()
    print(f'Weather data loaded: {len(weather)} weeks')
    print(weather.head())
except Exception as e:
    print(f'API not available in this environment: {e}')
    # Simulate weather data if API unavailable
    dates = pd.date_range('2019-01-01', periods=260, freq='W')
    temp = 10 + 15 * np.sin(2 * np.pi * np.arange(260) / 52) + np.random.normal(0, 2, 260)
    weather = pd.DataFrame({'temp_c': temp}, index=dates)
    weather.index.name = 'date'
    print('Using simulated weather data instead')

## 2. Build Demand Dataset with External Features

In [ ]:
np.random.seed(42)
n_weeks = 260
dates = pd.date_range(start='2019-01-01', periods=n_weeks, freq='W')

# Temperature-correlated demand (energy drinks sell more when warm)
temp_signal = weather['temp_c'].values[:n_weeks] if len(weather) >= n_weeks else \
    10 + 15 * np.sin(2 * np.pi * np.arange(n_weeks) / 52)

base_demand = 1000 + 1.5 * np.arange(n_weeks)
temp_effect = 8 * temp_signal  # 8 units per degree Celsius
noise = np.random.normal(0, 60, n_weeks)

# Simulate sporting event spikes (Red Bull sponsors many events)
event_weeks = [20, 32, 45, 72, 84, 110, 135, 160, 190, 215, 240]
event_boost = np.zeros(n_weeks)
for w in event_weeks:
    if w < n_weeks:
        event_boost[w] = np.random.uniform(200, 500)  # demand spike

demand = base_demand + temp_effect + event_boost + noise
demand = np.clip(demand, 0, None)

df = pd.DataFrame({
    'date': dates,
    'demand': demand,
    'temp_c': temp_signal,
    'has_event': (event_boost > 0).astype(int)
}).set_index('date')

print(f'Dataset: {len(df)} weeks | Demand range: {df.demand.min():.0f}–{df.demand.max():.0f}')

## 3. Feature Engineering with External Drivers

In [ ]:
def create_features_with_drivers(df):
    df = df.copy()

    # Standard lag features
    for lag in [1, 2, 4, 8, 52]:
        df[f'demand_lag_{lag}'] = df['demand'].shift(lag)

    # Rolling demand statistics
    df['rolling_mean_4']  = df['demand'].shift(1).rolling(4).mean()
    df['rolling_mean_12'] = df['demand'].shift(1).rolling(12).mean()

    # Calendar features
    df['week_of_year'] = df.index.isocalendar().week.astype(int)
    df['month']        = df.index.month
    df['is_summer']    = df['month'].isin([6, 7, 8]).astype(int)

    # External drivers (the key differentiators)
    df['temp_lag_1']     = df['temp_c'].shift(1)   # last week's temperature
    df['temp_rolling_4'] = df['temp_c'].shift(1).rolling(4).mean()
    # has_event is already in the dataframe

    return df.dropna()


df_f = create_features_with_drivers(df)

EXCLUDE = ['demand']
feature_cols = [c for c in df_f.columns if c not in EXCLUDE]
print(f'Total features: {len(feature_cols)}')
print(feature_cols)

## 4. Model Comparison — With vs Without External Drivers

In [ ]:
HORIZON = 12
X = df_f[feature_cols]
y = df_f['demand']

X_train, X_test = X.iloc[:-HORIZON], X.iloc[-HORIZON:]
y_train, y_test = y.iloc[:-HORIZON], y.iloc[-HORIZON:]

# Model A: without external drivers
base_features = [c for c in feature_cols
                 if not any(x in c for x in ['temp', 'event'])]
model_base = XGBRegressor(n_estimators=300, learning_rate=0.05,
                           max_depth=4, random_state=42)
model_base.fit(X_train[base_features], y_train)
pred_base = model_base.predict(X_test[base_features])

# Model B: with external drivers
model_full = XGBRegressor(n_estimators=300, learning_rate=0.05,
                           max_depth=4, random_state=42)
model_full.fit(X_train, y_train)
pred_full = model_full.predict(X_test)

mae_base = mean_absolute_error(y_test, pred_base)
mae_full = mean_absolute_error(y_test, pred_full)
improvement = (mae_base - mae_full) / mae_base * 100

print('=== Impact of External Demand Drivers ===')
print(f'MAE without drivers: {mae_base:.1f} units')
print(f'MAE with drivers:    {mae_full:.1f} units')
print(f'Improvement:         {improvement:.1f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(y_test.values, label='Actual demand', color='#2C2C2A', linewidth=1.8)
ax.plot(pred_base, label=f'Without drivers (MAE={mae_base:.0f})',
        color='#D85A30', linestyle='--', linewidth=1.4)
ax.plot(pred_full, label=f'With drivers (MAE={mae_full:.0f})',
        color='#185FA5', linestyle='--', linewidth=1.4)
ax.set_title('External Demand Drivers: Impact on Forecast Accuracy', fontsize=13)
ax.set_xlabel('Week')
ax.set_ylabel('Demand (units)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Key Takeaways

External demand drivers are the **biggest untapped opportunity** in most FMCG supply chains.
Most companies forecast purely from historical demand — ignoring signals that are known *in advance*.

**Weather is especially valuable for energy drinks:**
- Temperature forecasts are available 7–14 days ahead
- This means you can adjust your short-term supply plan *before* the demand spike arrives
- This is the difference between reactive and proactive supply chain management

**Next:** EUDR supplier risk pipeline (`03_eudr_supplier_risk/`) — connecting geospatial AI to supply chain.
